Use the SimLM model to help create a score based on the relevant data presented by the Ardook

Model Link: https://huggingface.co/intfloat/simlm-msmarco-reranker

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#Imports and Installation
!pip install transformers

import transformers
from transformers import AutoModelForSequenceClassification, AutoTokenizer, BatchEncoding, PreTrainedTokenizerFast
from transformers.modeling_outputs import SequenceClassifierOutput
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score


In [ ]:
def encode(tokenizer: PreTrainedTokenizerFast,
           query: str, passage: str, title: str = '-') -> BatchEncoding:
    return tokenizer(query,
                     text_pair='{}: {}'.format(title, passage),
                     max_length=192,
                     padding=True,
                     truncation=True,
                     return_tensors='pt')

tokenizer = AutoTokenizer.from_pretrained('intfloat/simlm-msmarco-reranker')
model = AutoModelForSequenceClassification.from_pretrained('intfloat/simlm-msmarco-reranker')
model.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

ElectraForSequenceClassification(
  (electra): ElectraModel(
    (embeddings): ElectraEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): ElectraEncoder(
      (layer): ModuleList(
        (0-11): 12 x ElectraLayer(
          (attention): ElectraAttention(
            (self): ElectraSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): ElectraSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (La

In [ ]:
FILE_PATH  = "/content/drive/MyDrive/Copy of Ardook_Data_All_Grant_Recommendations_UPDATED_v3.xlsx"   # path after uploading data
SHEET_NAME = "Grant Tracker"
TEXT_COL   = "Core Requirements"

In [ ]:
df = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME)
df.columns = df.columns.str.strip()
print(df.shape)
print("Empty requirement cells:", df[TEXT_COL].isna().sum())

# Creates the table
data = df[["Priority / Deadline Order", "Program", TEXT_COL, "Fit"]].copy()
data[TEXT_COL] = data[TEXT_COL].fillna("").astype(str).str.strip()
data.head()

(37, 13)
Empty requirement cells: 0


,Priority / Deadline Order,Program,Core Requirements,Fit
0,1,Women TechEU 2026–2028,Woman founder/co-founder in top management; wo...,Excellent
1,2,Tara’s Circle 2026,Women-led; international; seed-stage; defined/...,Excellent
2,3,Her Agenda Breakthrough Grant,Woman entrepreneur; operating business; newsle...,Strong
3,4,Hey Helen Grant (Visionaries),Woman; ≥50% ownership; U.S. resident; U.S. for...,Strong
4,5,WomensNet Amber Grant / Startup Grant,Women-owned business; Startup Grant track for ...,Strong


In [ ]:
#splits data so it's 80% train, 10% valiation, and 10% test
#train_df, temp_df = train_test_split(data, test_size=0.20, random_state=42)
#val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42)

# Verify the split sizes
#print(f"Total Dataset Size: {len(data)}")
#print(f"Train: {len(train_df)} ({len(train_df)/len(data):.1%})")
#print(f"Validation Set Size (10%): {len(val_df)} ({len(val_df)/len(data):.1%})")
#print(f"Test Set Size (10%): {len(test_df)} ({len(test_df)/len(data):.1%})")

In [ ]:
#Input Comparisons
scores = []
for index, row in data.iterrows():
  with torch.no_grad():
      bat_di = encode(tokenizer, 'List AI data and laundry grants available to small businesses in Armenia, and are at the early startup stage based on the fit', str(row[TEXT_COL]), str(row['Fit']))
      outputs: SequenceClassifierOutput = model(**bat_di, return_dict=True)
      results = torch.sigmoid(outputs.logits[0]).item() # torch.sigmoid = normalizes the output values (0 to 1) rather than -infinity to infinity --> raw data
      print(results)
      scores.append(results)

      #print(outputs.logits[0].item())
      #results.append(outputs.logits[0].item())

data['scores'] = scores


0.0003800392150878906
0.0003669261932373047
0.00038743019104003906
0.0003800392150878906
0.00046181678771972656
0.0003712177276611328
0.0003800392150878906
0.00037860870361328125
0.0004029273986816406
0.0004189014434814453
0.0003654956817626953
0.0003936290740966797
0.00037550926208496094
0.0004107952117919922
0.00043892860412597656
0.00044417381286621094
0.0003769397735595703
0.0003845691680908203
0.00037550926208496094
0.00037407875061035156
0.0003669261932373047
0.0003845691680908203
0.00038909912109375
0.0003845691680908203
0.00036406517028808594
0.00037407875061035156
0.0003726482391357422
0.0003936290740966797
0.00036978721618652344
0.0003726482391357422
0.00036835670471191406
0.0003669261932373047
0.0003981590270996094
0.0003769397735595703
0.0003905296325683594
0.0004222393035888672
0.0003814697265625


In [ ]:
# Converts the SimLM scores into percentage values
#data['absolute_percentage'] = data['scores'] * 100
maxval = data['scores'].max()
data['percentage'] = (data['scores'] / maxval) * 100

# Ranks the grants based on which one matches the prompt the most (predictions)
percentage_table = data[["Priority / Deadline Order", "Program", TEXT_COL, "Fit", "scores", "percentage"]].copy() #creates the table
percentage_table = percentage_table.sort_values(by='scores', ascending=False)

#percentage_table['absolute_percentage'] = percentage_table['absolute_percentage'].map('{:.2f}%'.format)
percentage_table['percentage'] = percentage_table['percentage'].map('{:.2f}%'.format)

table_values = percentage_table.head(10)

display(table_values) #illustrates the top 10 matches from the table

,Priority / Deadline Order,Program,Core Requirements,Fit,scores,percentage
4,5,WomensNet Amber Grant / Startup Grant,Women-owned business; Startup Grant track for ...,Strong,0.000462,100.00%
15,16,NVIDIA Inception,Incorporated startup; working website; at leas...,Strong,0.000444,96.18%
14,15,Accelerate California Innovation Grants,Innovative California small business; strategi...,Strong,0.000439,95.04%
35,36,Hello Alice + Google Main Street Rising AI Pitch,Small business owner; register/attend; selecte...,Moderate–Strong,0.000422,91.43%
9,10,EchoHer Founder Fellowship,Early-stage women/non-binary founders; selecti...,Excellent,0.000419,90.71%
13,14,Oasis Collective × AI After 5 – Women in AI Pi...,Women-led AI startup; investor-ready orientati...,Excellent,0.000411,88.95%
8,9,GTIA Grants,Nonprofit; referral/invitation via GTIA member...,Moderate,0.000403,87.25%
32,33,Humanity AI funding/program,Program-specific,Moderate,0.000398,86.22%
11,12,NSF America’s Seed Fund (SBIR/STTR Phase I),Qualifying U.S. small business; technically ri...,Strong if novel R&D; otherwise Moderate,0.000394,85.23%
27,28,DivInc Women in Tech Accelerator,Women-led early-stage tech/software; pilot/bet...,Strong thematic fit; low practical fit,0.000394,85.23%


In [ ]:
#Evaluation metrics --> compare top 10 grants to Ardook profile

#Baseline
ARDOOK = {
    "country": "Armenia",
    "legal_type": "for-profit company",
    "stage": "early-stage startup",
    "founded": 2023,
    "women_led": True,
    "focus": "data and AI tools for small businesses",
}

def profile_eval(row, profile):
    #Criteria = location, business type, primary audience
    valid_locale = profile["country"] == "Armenia"
    business_type = profile["legal_type"] == "for-profit company"
    primary_audience = profile["stage"] == "early-stage startup"

    criteria = str(row[TEXT_COL]).lower()
    if valid_locale:
        if "u.s. resident" in criteria or "us-only" in criteria or "united states only" in criteria:
            valid_locale = False
    if business_type:
        if "non-profit" in criteria and "for-profit" not in criteria and "business" not in criteria:
            business_type = False
    if primary_audience:
        stage_keywords = ["startup", "early-stage", "seed", "young", "new business", "founder"]
        if not any(keyword in criteria for keyword in stage_keywords):
            primary_audience = False
    if not valid_locale or not business_type or not primary_audience: #compares all three criteria together
        return 0
    return 1

#Passes the row and ardook profile w/ lambda
data['baseline_eval'] = data.apply(lambda r: profile_eval(r, ARDOOK), axis=1)
data['prediction'] = data['Program'].apply(lambda x: 1 if x in table_values['Program'].values else 0)

y_actual = data['baseline_eval']
y_pred = data['prediction']

#Accuracy
acc = accuracy_score(y_actual, y_pred)
print(f"Accuracy: {acc:.2%}")

#Precision
pre = precision_score(y_actual, y_pred)
print(f"Precision: {pre:.2%}")

#Recall
rec = recall_score(y_actual, y_pred)
print(f"Recall: {rec:.2%}")

Accuracy: 72.97%
Precision: 50.00%
Recall: 50.00%
